## Load Dataset

In [1]:
import pandas as pd

# 1. Load the CSVs
enron_df = pd.read_csv('./enron_folder/enron_spam_data.csv')
spam_assassin_df = pd.read_csv('./spamassassin_folder/spam_assassin.csv')
phishing_df = pd.read_csv('./phishing_folder/phishing_data.csv')
spam_and_ham_df = pd.read_csv('./spamandham_folder/spam_and_ham.csv')

## Data Preview

In [2]:
print("Enron Columns:", enron_df.columns.tolist())
print("SpamAssassin Columns:", spam_assassin_df.columns.tolist())
print("Phishing Columns:", phishing_df.columns.tolist())
print("SpamAndHam Columns:", spam_and_ham_df.columns.tolist())

print("--- ENRON PREVIEW ---")
display(enron_df.head(2))

print("\n--- SPAM ASSASSIN PREVIEW ---")
display(spam_assassin_df.head(2))

print("\n--- PHISHING PREVIEW ---")
display(phishing_df.head(2))

print("\n--- SPAM AND HAM PREVIEW ---")
display(spam_and_ham_df.head(2))

Enron Columns: ['Unnamed: 0', 'Subject', 'Message', 'Spam/Ham', 'Date']
SpamAssassin Columns: ['text', 'target']
Phishing Columns: ['text_combined', 'label']
SpamAndHam Columns: ['label', 'text']
--- ENRON PREVIEW ---


,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13



--- SPAM ASSASSIN PREVIEW ---


,text,target
0,From ilug-admin@linux.ie Mon Jul 29 11:28:02 2...,0
1,From gort44@excite.com Mon Jun 24 17:54:21 200...,1



--- PHISHING PREVIEW ---


,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0



--- SPAM AND HAM PREVIEW ---


,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...


## Combine Dataset

In [3]:
# --- 1. Process Enron ---
enron_final = enron_df[['Message', 'Spam/Ham']].copy()
enron_final.columns = ['email_text', 'risk_label']
enron_final['risk_label'] = enron_final['risk_label'].map({'ham': 0, 'spam': 1})

# --- 2. Process SpamAssassin ---
spam_assassin_final = spam_assassin_df[['text', 'target']].copy()
spam_assassin_final.columns = ['email_text', 'risk_label']

# --- 3. Process Phishing (Using CEAS_08.csv) ---
phishing_filtered = phishing_df[phishing_df['label'] == 1].copy()
phishing_final = phishing_filtered[['text_combined', 'label']].copy()
phishing_final.columns = ['email_text', 'risk_label']
phishing_final['risk_label'] = 2

# --- 4. Process the NEW SpamAndHam Dataset ---
spam_ham_final = spam_and_ham_df[['text', 'label']].copy()
spam_ham_final.columns = ['email_text', 'risk_label']

# --- 5. Combine EVERYTHING ---
master_df = pd.concat([enron_final, spam_assassin_final, phishing_final, spam_ham_final], ignore_index=True)

print("Distribution of Risk Tiers:")
print(master_df['risk_label'].value_counts())

# Check 2: Are there any empty emails?
print("\nMissing Values:")
print(master_df.isnull().sum())

# Check 3: Peek at the final product
display(master_df.sample(5)) # Shows 5 random rows from the mix

Distribution of Risk Tiers:
risk_label
1    62977
0    59983
2    42891
Name: count, dtype: int64

Missing Values:
email_text    52
risk_label     0
dtype: int64


,email_text,risk_label
79231,mr david bah d_bah15yahoocom greeting sir cont...,2
40024,bigger breast pill image loading cli k info sh...,2
17789,start date : 1 / 20 / 02 ; hourahead hour : 18...,0
7363,"thanks a lot ,\nand just let me know the sched...",0
39703,loading messages,2


## Data Cleaning & Preprocessing

### Remove empty row

In [4]:
# Drop rows where the email text is missing
master_df = master_df.dropna(subset=['email_text']).reset_index(drop=True)

# Run the check again just to be sure!
print("Missing Values after cleaning:\n", master_df.isnull().sum())

Missing Values after cleaning:
 email_text    0
risk_label    0
dtype: int64


### Noise Reduction

In [5]:
import re

def clean_text(text):
    # 1. Convert to lowercase
    text = str(text).lower()
    
    # 2. Replace link with placeholder
    text = re.sub(r'http\S+|www\S+|https\S+', ' url_link ', text)
    
    # 3. Remove HTML tags (like <br>, <html>)
    text = re.sub(r'<.*?>', '', text)
    
    # 4. Remove special characters, punctuation, and newline tags (\n, \t)
    # This keeps only letters and numbers
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # 5. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply the cleaning function to your entire dataset (This might take a minute or two to run!)
print("Cleaning text data... please wait.")
master_df['clean_text'] = master_df['email_text'].apply(clean_text)

print("Cleaning complete!")
display(master_df[['email_text', 'clean_text', 'risk_label']].head())

Cleaning text data... please wait.
Cleaning complete!


,email_text,clean_text,risk_label
0,"gary , production from the high island larger ...",gary production from the high island larger bl...,0
1,- calpine daily gas nomination 1 . doc,calpine daily gas nomination 1 doc,0
2,fyi - see note below - already done .\nstella\...,fyi see note below already done stella forward...,0
3,fyi .\n- - - - - - - - - - - - - - - - - - - -...,fyi forwarded by lauri a allen hou ect on 12 1...,0
4,"jackie ,\nsince the inlet to 3 river plant is ...",jackie since the inlet to 3 river plant is shu...,0


### Remove Duplicate

In [6]:
initial_rows = len(master_df) # Count before dropping

# Drop the duplicates based on the text column
master_df = master_df.drop_duplicates(subset=['clean_text'])

final_rows = len(master_df) # Count after dropping
removed_count = initial_rows - final_rows # Calculate how many were removed

# Print the results you asked for
print(f"Duplicates removed: {removed_count}")
print(f"Total rows remaining after removal: {final_rows}")

print("New Distribution after dropping duplicates:")
print(master_df['risk_label'].value_counts())

Duplicates removed: 34764
Total rows remaining after removal: 131035
New Distribution after dropping duplicates:
risk_label
1    45397
2    42831
0    42807
Name: count, dtype: int64


### Import Library

In [7]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/macintosh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/macintosh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/macintosh/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Tokenization

In [8]:
def tokenize_text(text):
    return word_tokenize(str(text))

master_df['step1_tokens'] = master_df['clean_text'].apply(tokenize_text)

print("Tokenization Complete!")
display(master_df[['clean_text', 'step1_tokens']].head(3))

Tokenization Complete!


,clean_text,step1_tokens
0,gary production from the high island larger bl...,"[gary, production, from, the, high, island, la..."
1,calpine daily gas nomination 1 doc,"[calpine, daily, gas, nomination, 1, doc]"
2,fyi see note below already done stella forward...,"[fyi, see, note, below, already, done, stella,..."


### Stopwords removal

In [9]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(token_list):
    return [word for word in token_list if word not in stop_words]

master_df['step2_no_stopwords'] = master_df['step1_tokens'].apply(remove_stopwords)

print("Stopwords Removal Complete!")
display(master_df[['step1_tokens', 'step2_no_stopwords']].head(3))

Stopwords Removal Complete!


,step1_tokens,step2_no_stopwords
0,"[gary, production, from, the, high, island, la...","[gary, production, high, island, larger, block..."
1,"[calpine, daily, gas, nomination, 1, doc]","[calpine, daily, gas, nomination, 1, doc]"
2,"[fyi, see, note, below, already, done, stella,...","[fyi, see, note, already, done, stella, forwar..."


### Lemmatization

In [10]:
lemmatizer = WordNetLemmatizer()

def lemmatize_and_join(token_list):
    lemmatized = [lemmatizer.lemmatize(word) for word in token_list]
    return ' '.join(lemmatized) # Join with spaces

master_df['final_nlp_text'] = master_df['step2_no_stopwords'].apply(lemmatize_and_join)

print("Lemmatization Complete!")
display(master_df[['step2_no_stopwords', 'final_nlp_text', 'risk_label']].head(3))

Lemmatization Complete!


,step2_no_stopwords,final_nlp_text,risk_label
0,"[gary, production, high, island, larger, block...",gary production high island larger block 1 2 c...,0
1,"[calpine, daily, gas, nomination, 1, doc]",calpine daily gas nomination 1 doc,0
2,"[fyi, see, note, already, done, stella, forwar...",fyi see note already done stella forwarded ste...,0


In [11]:
final_export_df = master_df[['final_nlp_text', 'risk_label']].copy()

final_export_df = final_export_df.dropna()
final_export_df = final_export_df[final_export_df['final_nlp_text'].str.strip() != '']

file_name = 'cleaned_email_dataset.csv'
final_export_df.to_csv(file_name, index=False)

print(f"Success")

Success


## MODEL TRAINING

In [12]:
import pandas as pd
import numpy as np
import joblib
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
 
import matplotlib.pyplot as plt
 
print("All libraries imported successfully!")

All libraries imported successfully!


In [13]:
df = pd.read_csv('cleaned_email_dataset.csv')
 
# Safety checks
df = df.dropna(subset=['final_nlp_text', 'risk_label'])
df = df[df['final_nlp_text'].str.strip() != '']
df['risk_label'] = df['risk_label'].astype(int)
 
# Labels: 0 = Ham (Low Risk), 1 = Spam (Mid Risk), 2 = Phishing (High Risk)
label_names = {0: 'Ham (Low Risk)', 1: 'Spam (Mid Risk)', 2: 'Phishing (High Risk)'}
 
print(f"Dataset loaded: {len(df):,} rows")
print("\nClass distribution:")
print(df['risk_label'].value_counts().sort_index().rename(label_names))

Dataset loaded: 131,026 rows

Class distribution:
risk_label
Ham (Low Risk)          42805
Spam (Mid Risk)         45390
Phishing (High Risk)    42831
Name: count, dtype: int64


In [14]:
X = df['final_nlp_text']
y = df['risk_label']
 
# stratify=y ensures all 3 classes appear proportionally in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
 
print(f"Training set : {len(X_train):,} samples")
print(f"Testing set  : {len(X_test):,} samples")
print("\nTraining class distribution:")
print(y_train.value_counts().sort_index().rename(label_names))

Training set : 104,820 samples
Testing set  : 26,206 samples

Training class distribution:
risk_label
Ham (Low Risk)          34244
Spam (Mid Risk)         36312
Phishing (High Risk)    34264
Name: count, dtype: int64


In [15]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=100_000,
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
    analyzer='word',
)
 
# IMPORTANT: fit ONLY on training data to prevent data leakage
print("Fitting TF-IDF on training data...")
X_train_tfidf = tfidf.fit_transform(X_train)
 
# Transform test data using the already-fitted vectorizer
X_test_tfidf = tfidf.transform(X_test)
 
print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"  → {X_train_tfidf.shape[0]:,} emails × {X_train_tfidf.shape[1]:,} features")
 
# Save the vectorizer NOW (Person 4 needs this to transform new emails)
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("\n✅ Saved: tfidf_vectorizer.pkl")

Fitting TF-IDF on training data...
TF-IDF matrix shape (train): (104820, 100000)
  → 104,820 emails × 100,000 features

✅ Saved: tfidf_vectorizer.pkl


In [ ]:
svm = SVC(
    kernel='linear',
    probability=True,
    class_weight='balanced',
    random_state=42
)
 
# Grid of C values to try (C controls regularisation strength)
# Smaller C = more regularisation (simpler model)
# Larger C = less regularisation (fits training data more tightly)
param_grid = {
    'C': [0.1, 1, 10]
}
 
# StratifiedKFold preserves class ratio in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',   # macro-F1 gives equal weight to all 3 classes
    n_jobs=1,            # use all CPU cores to speed things up
    verbose=2,
    refit=True            # auto-refit best model on full training set
)
 
print("Starting GridSearchCV... (this will take a while)")
grid_search.fit(X_train_tfidf, y_train)
 
print(f"\n✅ Best C value  : {grid_search.best_params_['C']}")
print(f"   Best macro-F1 : {grid_search.best_score_:.4f}")
 
best_svm = grid_search.best_estimator_

In [ ]:
y_pred  = best_svm.predict(X_test_tfidf)
y_proba = best_svm.predict_proba(X_test_tfidf)  # shape: (n_samples, 3)
 
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")
 
target_names = [label_names[i] for i in sorted(label_names)]
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names).plot(
    ax=ax, cmap='Blues', colorbar=False
)
ax.set_title('Confusion Matrix — Email Risk Classifier', fontsize=13, pad=12)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
print("=== predict_proba demo on 3 test samples ===\n")
 
for i in range(3):
    text    = X_test.iloc[i]
    true    = y_test.iloc[i]
    proba   = y_proba[i]          # array of 3 probabilities summing to 1.0
    pred    = y_pred[i]
 
    print(f"Sample {i+1}:")
    print(f"  Snippet      : {text[:80]}...")
    print(f"  True label   : {label_names[true]}")
    print(f"  Predicted    : {label_names[pred]}")
    print(f"  Probabilities:")
    for class_idx, prob in enumerate(proba):
        bar = '█' * int(prob * 30)
        print(f"    {label_names[class_idx]:25s} {prob*100:5.1f}%  {bar}")
    print()

In [ ]:
joblib.dump(best_svm, 'email_model.pkl')
print("✅ Saved: email_model.pkl")
print()
print("=== PERSON 3 DELIVERABLES COMPLETE ===")
print("  tfidf_vectorizer.pkl  → hand this to Person 4")
print("  email_model.pkl       → hand this to Person 4")
print("  confusion_matrix.png  → use in presentation slides (Person 1)")